In [ ]:
import os
import sys

CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Locals
from utils import *
from dataset import Dataset
from models import *
from dlnr import DLNoiseReduction

In [ ]:
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH, 'Informer_Datasets', 'ETT', 'ETTh1.csv'))
df.tail()

In [ ]:
df.index = pd.to_datetime(df['date'])
df.drop(columns=['date'], inplace=True)
df.describe().T

In [ ]:
X = df[df.columns[:-1]]
y = df[df.columns[-1:]]

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.describe().T

In [ ]:
# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
X_train = torch.tensor(X_train.values, dtype=torch.float32).to(device)
X_val = torch.tensor(X_val.values, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

In [ ]:
# Agregar una dimensión adicional para los timesteps (necesario para GRU)
X_train = X_train.unsqueeze(1)  # Añadir dimensión para "timesteps"
X_val = X_val.unsqueeze(1)
X_test = X_test.unsqueeze(1)

In [ ]:
input_size = len(X_scaled.columns)  # Número de variables de entrada
hidden_size = 50  # Número de unidades en la GRU
output_size = len(y.columns)  # Predicción de una variable
num_layers = 1  # Una sola capa GRU

# Crear el modelo
model = GRUModel(input_size, hidden_size, output_size, num_layers).to(device)

# Definir la función de pérdida y el optimizador
criterion = nn.MSELoss()  # Error cuadrático medio
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
train_dataloader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_dataloader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)
test_dataloader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)

In [ ]:
trainer = Trainer(
    model = model,
    train_generator = train_dataloader,
    val_generator = val_dataloader,
    device = device,
    criterion = criterion,
    optimizer = optimizer,
    epoch_scheduler = None,
    batch_scheduler = None,
    patience = 10,
    epochs = 100,
    checkpoints_path = os.path.join(CHECKPOINT_PATH, 'ETT', 'gru.pth'),
)

In [ ]:
trainer.fit(verbose=True)

In [ ]:
predictions = trainer.eval_dataloader(test_dataloader)

In [ ]:
predictions = torch.cat(predictions, dim=0)
predictions.shape

In [ ]:
np.sqrt(criterion(predictions, y_test).cpu().detach().numpy())